# CRM Reactivation — Baseline Notebook

REGION missing → fill with "Unknown"

CANCELLED == 'X' → exclude from valid activity/features

negative NET → exclude

clients with only cancelled transactions → exclude from modeling

categorical variables → one-hot encode

model → logistic regression with class_weight="balanced" as the simplest imbalance handling

In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score
)

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

load dataset

In [2]:
client_df = pd.read_csv("TOOL_CLIENT.csv")
sales_df = pd.read_csv("TOOL_SALES.csv")

In [3]:
print("TOOL_CLIENT shape:", client_df.shape)
print("TOOL_SALES shape:", sales_df.shape)

display(client_df.head())
display(sales_df.head())

display(client_df.info())
display(sales_df.info())

TOOL_CLIENT shape: (93257, 8)
TOOL_SALES shape: (2050449, 10)


,CLIENT_ID,CLIENT_CREATE DATE,REGION,TRADE SECTOR,N_EMPLOYEES,ECONOMIC_POT,ECO_POT_CLASS,RISK_CAT
0,9306,2005-11-15 00:00:00,BZ,11000,6,"8,659.810",D,3d
1,939,2005-11-15 00:00:00,LE,15500,2,681.260,E,3d
2,8321,2005-11-15 00:00:00,LE,15500,2,681.260,E,T8
3,4174,2005-11-15 00:00:00,BZ,15400,1,494.450,E,3d
4,12765,2005-11-15 00:00:00,BZ,15400,1,494.450,E,3b


,YYYYMM,ITEM_ID,FLG_TOOL,SALES_CHANNEL,NET,UNIT,FAMILY_CODE,GROUP_CODE,CLIENT_ID,CANCELLED
0,201701,54,0,B,24.780,P,XBAM2HA,XBAM2HA0201,15674,NaN
1,201701,44,0,B,34.620,P,XBAM2HA,XBAM2HA0201,62486,NaN
2,201701,47,0,D,46.450,P,XBAM2HA,XBAM2HA0201,60149,NaN
3,201701,52,0,B,62.150,P,XBAM2HA,XBAM2HA0201,42371,NaN
4,201701,56,0,B,35.800,P,XBAM2HA,XBAM2HA0201,13144,NaN


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 93257 entries, 0 to 93256
Data columns (total 8 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   CLIENT_ID           93257 non-null  int64  
 1   CLIENT_CREATE DATE  93257 non-null  object 
 2   REGION              91276 non-null  object 
 3   TRADE SECTOR        93257 non-null  int64  
 4   N_EMPLOYEES         93257 non-null  int64  
 5   ECONOMIC_POT        93257 non-null  float64
 6   ECO_POT_CLASS       93257 non-null  object 
 7   RISK_CAT            93257 non-null  object 
dtypes: float64(1), int64(3), object(4)
memory usage: 5.7+ MB


None

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2050449 entries, 0 to 2050448
Data columns (total 10 columns):
 #   Column         Dtype  
---  ------         -----  
 0   YYYYMM         int64  
 1   ITEM_ID        int64  
 2   FLG_TOOL       int64  
 3   SALES_CHANNEL  object 
 4   NET            float64
 5   UNIT           object 
 6   FAMILY_CODE    object 
 7   GROUP_CODE     object 
 8   CLIENT_ID      int64  
 9   CANCELLED      object 
dtypes: float64(1), int64(4), object(5)
memory usage: 156.4+ MB


None

In [4]:
# --- 3A. Inspect raw client data quality ---
print("Missing values:")
display(client_df.isna().sum().sort_values(ascending=False))

print("\nDuplicate CLIENT_ID:", client_df["CLIENT_ID"].duplicated().sum())

for col in ["REGION", "TRADE SECTOR", "ECO_POT_CLASS", "RISK_CAT"]:
    print(f"\nValue counts for {col}:")
    display(client_df[col].value_counts(dropna=False).head(10))

Missing values:


REGION                1981
CLIENT_ID                0
CLIENT_CREATE DATE       0
TRADE SECTOR             0
N_EMPLOYEES              0
ECONOMIC_POT             0
ECO_POT_CLASS            0
RISK_CAT                 0
dtype: int64


Duplicate CLIENT_ID: 0

Value counts for REGION:


REGION
RM     4768
MI     3968
TO     2755
BO     2492
FI     2480
BS     2035
NaN    1981
TV     1923
GE     1895
VR     1857
Name: count, dtype: int64


Value counts for TRADE SECTOR:


TRADE SECTOR
11000    35687
21100     9829
13500     8893
22100     7169
42200     4531
15100     3167
22000     2731
14400     2191
13310     1556
15400     1520
Name: count, dtype: int64


Value counts for ECO_POT_CLASS:


ECO_POT_CLASS
E    57725
D    35532
Name: count, dtype: int64


Value counts for RISK_CAT:


RISK_CAT
3d    30918
2a    17417
5d    14471
3a    13537
T8     9420
4d     2707
5a     2474
3b      616
2d      318
4c      281
Name: count, dtype: int64

In [ ]:
client_clean = client_df.copy()


In [7]:
# date
#errors="coerce"
# If a value cannot be parsed as a valid date, pandas does not raise an error.
# Instead, it replaces the invalid value with NaT (“Not a Time”, the datetime equivalent of NaN).
client_clean["CLIENT_CREATE DATE"] = pd.to_datetime(
    client_clean["CLIENT_CREATE DATE"],
    errors="coerce"
)

In [ ]:
# This returns all rows where the column is NaT.
invalid_dates = client_clean[client_clean["CLIENT_CREATE DATE"].isna()]
print(invalid_dates)
# result: all date columns ok

Empty DataFrame
Columns: [CLIENT_ID, CLIENT_CREATE DATE, REGION, TRADE SECTOR, N_EMPLOYEES, ECONOMIC_POT, ECO_POT_CLASS, RISK_CAT]
Index: []


In [ ]:
# simple missing handling for REGION
client_clean["REGION"] = client_clean["REGION"].fillna("Unknown")
(client_clean["REGION"] == "Unknown").sum()

# !!!!!!!! 1981 Unknown rgion valus

1981

In [17]:
# numeric columns
for col in ["N_EMPLOYEES", "ECONOMIC_POT", "TRADE SECTOR"]:
    client_clean[col] = pd.to_numeric(client_clean[col], errors="coerce")
# If a value cannot be converted to a number, it becomes NaN instead of raising an error.

# Check for missing values after conversion
print(client_clean["N_EMPLOYEES"].isna().sum())
print(client_clean["N_EMPLOYEES"].isna().sum())
print(client_clean["N_EMPLOYEES"].isna().sum())


0
0
0


In [ ]:
# optional: make some categoricals explicit
for col in ["REGION", "ECO_POT_CLASS", "RISK_CAT"]:
    client_clean[col] = client_clean[col].astype("category")

# A categorical column stores:
# a list of unique categories
# the data as integer codes referencing those categories

In [ ]:
print("Cleaned TOOL_CLIENT shape:", client_clean.shape)
print("\nMissing values after cleaning:")
display(client_clean.isna().sum().sort_values(ascending=False))

display(client_clean.head())

Cleaned TOOL_CLIENT shape: (93257, 8)

Missing values after cleaning:


CLIENT_ID             0
CLIENT_CREATE DATE    0
REGION                0
TRADE SECTOR          0
N_EMPLOYEES           0
ECONOMIC_POT          0
ECO_POT_CLASS         0
RISK_CAT              0
dtype: int64

,CLIENT_ID,CLIENT_CREATE DATE,REGION,TRADE SECTOR,N_EMPLOYEES,ECONOMIC_POT,ECO_POT_CLASS,RISK_CAT
0,9306,2005-11-15,BZ,11000,6,"8,659.810",D,3d
1,939,2005-11-15,LE,15500,2,681.260,E,3d
2,8321,2005-11-15,LE,15500,2,681.260,E,T8
3,4174,2005-11-15,BZ,15400,1,494.450,E,3d
4,12765,2005-11-15,BZ,15400,1,494.450,E,3b


,CLIENT_ID,CLIENT_CREATE DATE,REGION,TRADE SECTOR,N_EMPLOYEES,ECONOMIC_POT,ECO_POT_CLASS,RISK_CAT
count,"93,257.000",93257,93257,"93,257.000","93,257.000","93,257.000",93257,93257
unique,NaN,NaN,122,NaN,NaN,NaN,2,20
top,NaN,NaN,RM,NaN,NaN,NaN,E,3d
freq,NaN,NaN,4768,NaN,NaN,NaN,57725,30918
mean,"46,629.000",2012-05-20 16:31:58.414703616,NaN,"17,519.570",5.428,"4,942.024",NaN,NaN
min,1.000,2005-11-15 00:00:00,NaN,"10,000.000",1.000,0.000,NaN,NaN
25%,"23,315.000",2005-11-16 00:00:00,NaN,"11,000.000",1.000,"2,264.230",NaN,NaN
50%,"46,629.000",2012-02-21 00:00:00,NaN,"13,500.000",3.000,"4,528.530",NaN,NaN
75%,"69,943.000",2017-09-22 00:00:00,NaN,"21,100.000",5.000,"7,528.520",NaN,NaN
max,"93,257.000",2021-12-23 00:00:00,NaN,"42,500.000","10,000.000","21,069.210",NaN,NaN


Let's take a look at data

In [ ]:

display(client_clean.describe(include="all"))

In [ ]:
print("Raw sales shape:", sales_df.shape)

print("\nMissing values:")
display(sales_df.isna().sum().sort_values(ascending=False))

print("\nCANCELLED counts:")
display(sales_df["CANCELLED"].value_counts(dropna=False))

print("\nNET summary:")
display(sales_df["NET"].describe())

print("\nNegative NET rows:", (pd.to_numeric(sales_df["NET"], errors="coerce") < 0).sum())


Raw sales shape: (2050449, 10)

Missing values:


CANCELLED        1947739
YYYYMM                 0
ITEM_ID                0
FLG_TOOL               0
SALES_CHANNEL          0
NET                    0
UNIT                   0
FAMILY_CODE            0
GROUP_CODE             0
CLIENT_ID              0
dtype: int64


CANCELLED counts:


CANCELLED
NaN    1947739
X       102710
Name: count, dtype: int64


NET summary:


count   2,050,449.000
mean          116.850
std           227.334
min       -13,132.970
25%            13.680
50%            49.090
75%           135.780
max        59,541.920
Name: NET, dtype: float64


Negative NET rows: 2298

Sample YYYYMM values:


0    201701
1    201701
2    201701
3    201701
4    201701
5    201701
6    201701
7    201701
8    201701
9    201701
Name: YYYYMM, dtype: int64

# CLeaniing second dataseeet

In [22]:
sales_clean = sales_df.copy()

# Parse YYYYMM as first day of month
sales_clean["purchase_month"] = pd.to_datetime(
    sales_clean["YYYYMM"].astype(str) + "01",
    format="%Y%m%d",
    errors="coerce"
)

# Numeric coercion
sales_clean["NET"]      = pd.to_numeric(sales_clean["NET"],      errors="coerce")
sales_clean["FLG_TOOL"] = pd.to_numeric(sales_clean["FLG_TOOL"], errors="coerce")

# Standardize CANCELLED a bit for safety
sales_clean["CANCELLED"] = sales_clean["CANCELLED"].fillna("").astype(str).str.strip()

# Valid purchases for target/features:
# - not cancelled
# - positive NET only
sales_valid = sales_clean[(sales_clean["CANCELLED"] != "X") & (sales_clean["NET"] > 0)].copy()

print("sales_clean shape:", sales_clean.shape)
print("sales_valid shape:", sales_valid.shape)

print("\nDate range in sales_valid:")
print(sales_valid["purchase_month"].min(), "to", sales_valid["purchase_month"].max())

print("\nRemaining missing values in key fields:")

display(sales_valid[["CLIENT_ID", "purchase_month", "NET"]].isna().sum())

sales_clean shape: (2050449, 11)
sales_valid shape: (1617609, 11)

Date range in sales_valid:
2017-01-01 00:00:00 to 2021-12-01 00:00:00

Remaining missing values in key fields:


CLIENT_ID         0
purchase_month    0
NET               0
dtype: int64

Clients that have only cancelled, never purchased anything

In [23]:
clients_all_sales   = set(sales_clean["CLIENT_ID"].dropna().unique())
clients_valid_sales = set(sales_valid["CLIENT_ID"].dropna().unique())

cancel_only_clients = clients_all_sales - clients_valid_sales

print("Clients with any sales rows:", len(clients_all_sales))
print("Clients with at least one valid purchase:", len(clients_valid_sales))
print("Cancel-only clients:", len(cancel_only_clients))

Clients with any sales rows: 93257
Clients with at least one valid purchase: 85361
Cancel-only clients: 7896


target construction

Baseline target for first notebook:
- inactive pool = clients with no valid purchase from start of 2017 to the end of 2018
- target = 1 if they purchase anything from beginning of 2019 to end of 2020, else 0

windows

In [ ]:
client_clean["CLIENT_CREATE DATE"].min()
client_clean["CLIENT_CREATE DATE"].max()

# sales_clean["YYYYMM"].min() #201701
sales_clean["YYYYMM"].max()   #202112



202112

In [ ]:
cutoff_date = pd.Timestamp("2018-12-31")

min_registration_date = pd.Timestamp("2016-12-31")

feature_start = pd.Timestamp("2017-01-01")
feature_end = pd.Timestamp("2018-12-31")

outcome_start = pd.Timestamp("2019-01-01")
outcome_end = pd.Timestamp("2020-12-31")

# !!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
# !!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
# !!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
# !!!!!!!!!!!!!!!!!!!!!!!!!!IMPORTANT!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
# !!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
# !!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
# !!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
# 1) only clients old enough to possibly be 24m inactive by end-2018
eligible_registry_clients = set(
    client_clean.loc[
        client_clean["CLIENT_CREATE DATE"] <= min_registration_date,
        "CLIENT_ID"
    ].unique()
)

#The eLIGIBLe clients are those that registered on or before 2016-12-31, 
# which means they could have been inactive for the full 24 months of 2017-2018 if they had
# no valid purchases during that period. This ensures we are only considering clients who had the
# opportunity to be inactive for the entire feature period, making our target variable meaningful
# and avoiding bias from newer clients who couldn't have been inactive for the full duration.
# 2) remove clients with no valid purchase behavior at all
eligible_clients = eligible_registry_clients - cancel_only_clients

# 3) clients active in 2017-2018
# These clients are not part of the inactive pool, 
# but we will check if they reactivate in 2019-2020 to understand the behavior of active vs 
# inactive clients.
clients_active_2017_2018 = set(
    sales_valid.loc[
        sales_valid["purchase_month"].between(feature_start, feature_end),
        "CLIENT_ID"
    ].unique()
)

# 4) inactive pool
inactive_clients_2018 = eligible_clients - clients_active_2017_2018

# Note: some of these "inactive" clients might have been reactivated during 2017-2018,
# but we are defining inactivity based on the feature period to avoid data leakage. We will 
# check for reactivation in the outcome period separately.
# 5) reactivated in 2019-2020
clients_reactivated_2019_2020 = set(
    sales_valid.loc[
        sales_valid["purchase_month"].between(outcome_start, outcome_end),
        "CLIENT_ID"
    ].unique()
)
# some of these reactivated clients might have been active in 2017-2018, but we are interested in
# whether any of the inactive pool reactivated, so we will check the overlap with the 
# inactive_clients_2018 set.

target_df = pd.DataFrame({"CLIENT_ID": list(inactive_clients_2018)})
target_df["target"] = target_df["CLIENT_ID"].isin(clients_reactivated_2019_2020).astype(int)

print("Eligible registry clients (created <= 2016-12-31):", len(eligible_registry_clients))
print("Eligible clients after removing cancel-only:", len(eligible_clients))
print("Inactive pool at end of 2018:", len(target_df))
print("\nTarget distribution:")
display(target_df)
print("\nReactivation rate:", target_df["target"].mean())
# This is the sum of  the target =  1 divided by the total number of clients in the inactive pool, 
# giving us the percentage of those clients who reactivated in 2019-2020.

Eligible registry clients (created <= 2016-12-31): 64690
Eligible clients after removing cancel-only: 58232
Inactive pool at end of 2018: 10865

Target distribution:


,CLIENT_ID,target
0,1,1
1,5,1
2,32775,1
3,10,1
4,14,0
...,...,...
10860,32724,1
10861,32732,1
10862,32734,1
10863,32743,1



Reactivation rate: 0.7554532903819604


In [25]:
# This should be zero now
post_cutoff_clients_in_target = (
    client_clean.merge(target_df, on="CLIENT_ID", how="inner")
               .query("`CLIENT_CREATE DATE` > '2016-12-31'")
               .shape[0]
)

print("Clients in target created after 2016-12-31:", post_cutoff_clients_in_target)

Clients in target created after 2016-12-31: 0


In [26]:
print("Unique CLIENT_ID in target_df:", target_df["CLIENT_ID"].nunique())
print("Any duplicated CLIENT_ID?", target_df["CLIENT_ID"].duplicated().sum())
display(target_df.head())

Unique CLIENT_ID in target_df: 10865
Any duplicated CLIENT_ID? 0


,CLIENT_ID,target
0,1,1
1,5,1
2,32775,1
3,10,1
4,14,0


So for this first basic notebook, don’t force transaction features yet. A simple and defensible baseline is:

keep this target as an approximate label

use only client-registry features

fit logistic regression

clearly note the target limitation in markdown

In [27]:
# Merge target with client registry
model_df = target_df.merge(client_clean, on="CLIENT_ID", how="left")

# Keep only the simplest baseline features from TOOL_CLIENT
feature_cols = [
    "REGION",
    "TRADE SECTOR",
    "N_EMPLOYEES",
    "ECONOMIC_POT",
    "ECO_POT_CLASS",
    "RISK_CAT"
]

X = model_df[feature_cols].copy()
y = model_df["target"].copy()

print("Modeling table shape:", model_df.shape)
print("Feature matrix shape:", X.shape)
print("Target mean:", y.mean())

display(model_df.head())
display(X.dtypes)

Modeling table shape: (10865, 9)
Feature matrix shape: (10865, 6)
Target mean: 0.7554532903819604


,CLIENT_ID,target,CLIENT_CREATE DATE,REGION,TRADE SECTOR,N_EMPLOYEES,ECONOMIC_POT,ECO_POT_CLASS,RISK_CAT
0,1,1,2005-11-15,VT,22100,2,"4,528.460",E,3d
1,5,1,2005-11-15,AN,13500,2,"4,432.420",E,3d
2,32775,1,2007-09-14,MI,13500,4,"5,000.000",E,5d
3,10,1,2005-11-15,PN,21100,1,"7,000.000",D,5a
4,14,0,2005-11-15,PO,13310,3,"3,291.900",E,3d


REGION           category
TRADE SECTOR        int64
N_EMPLOYEES         int64
ECONOMIC_POT      float64
ECO_POT_CLASS    category
RISK_CAT         category
dtype: object

In [28]:
print("Missing values in X:")
display(X.isna().sum())

print("\nTarget distribution:")
display(y.value_counts(normalize=True))

Missing values in X:


REGION           0
TRADE SECTOR     0
N_EMPLOYEES      0
ECONOMIC_POT     0
ECO_POT_CLASS    0
RISK_CAT         0
dtype: int64


Target distribution:


target
1   0.755
0   0.245
Name: proportion, dtype: float64

## 7. Baseline model: logistic regression

Baseline choices:
- registry features only
- add client age at cutoff
- stratified train/test split
- one-hot encode categoricals
- scale numeric features
- plain logistic regression

add  clint agw

In [29]:
# Age of client at the inactivity cutoff
model_df["client_age_months"] = (
    (cutoff_date.year - model_df["CLIENT_CREATE DATE"].dt.year) * 12
    + (cutoff_date.month - model_df["CLIENT_CREATE DATE"].dt.month)
)

feature_cols = [
    "REGION",
    "TRADE SECTOR",
    "N_EMPLOYEES",
    "ECONOMIC_POT",
    "ECO_POT_CLASS",
    "RISK_CAT",
    "client_age_months"
]

X = model_df[feature_cols].copy()
y = model_df["target"].copy()

display(X.head())

,REGION,TRADE SECTOR,N_EMPLOYEES,ECONOMIC_POT,ECO_POT_CLASS,RISK_CAT,client_age_months
0,VT,22100,2,"4,528.460",E,3d,157
1,AN,13500,2,"4,432.420",E,3d,157
2,MI,13500,4,"5,000.000",E,5d,135
3,PN,21100,1,"7,000.000",D,5a,157
4,PO,13310,3,"3,291.900",E,3d,157


In [31]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("Train target mean:", y_train.mean())
print("Test target mean:", y_test.mean())

X_train: (8692, 7)
X_test: (2173, 7)
Train target mean: 0.7554072710538426
Test target mean: 0.7556373676944317


In [19]:
numeric_features = [
    "TRADE SECTOR",
    "N_EMPLOYEES",
    "ECONOMIC_POT",
    "client_age_months"
]

categorical_features = [
    "REGION",
    "ECO_POT_CLASS",
    "RISK_CAT"
]

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

clf = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=2000, random_state=42))
])

clf.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['TRADE SECTOR',
                                                   'N_EMPLOYEES',
                                                   'ECONOMIC_POT',
                                                   'client_age_months']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['REGION', 'ECO_POT_CLASS',
                                                   'RISK_CAT'])])),
                ('model', LogisticRegression(max_iter=2000, random_state=42))])

In [32]:
y_pred = clf.predict(X_test)
y_proba = clf.predict_proba(X_test)[:, 1]

print("AUC:", roc_auc_score(y_test, y_proba))
print("\nClassification report:")
print(classification_report(y_test, y_pred))

print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred))

AUC: 0.7545727616177048

Classification report:
              precision    recall  f1-score   support

           0       0.71      0.23      0.35       531
           1       0.80      0.97      0.87      1642

    accuracy                           0.79      2173
   macro avg       0.75      0.60      0.61      2173
weighted avg       0.77      0.79      0.75      2173

Confusion matrix:
[[ 124  407]
 [  51 1591]]


In [33]:
import pandas as pd
import numpy as np

# Get transformed feature names from the fitted preprocessor
feature_names = clf.named_steps["preprocessor"].get_feature_names_out()

# Get logistic regression coefficients
coefs = clf.named_steps["model"].coef_[0]

coef_df = pd.DataFrame({
    "feature": feature_names,
    "coefficient": coefs,
    "abs_coefficient": np.abs(coefs),
    "odds_ratio": np.exp(coefs)
}).sort_values("abs_coefficient", ascending=False)

display(coef_df.head(20))

,feature,coefficient,abs_coefficient,odds_ratio
125,cat__RISK_CAT_3d,1.646,1.646,5.188
130,cat__RISK_CAT_5a,-1.610,1.610,0.200
126,cat__RISK_CAT_4a,-1.006,1.006,0.366
132,cat__RISK_CAT_5c,-0.961,0.961,0.383
129,cat__RISK_CAT_4d,0.956,0.956,2.602
133,cat__RISK_CAT_5d,0.893,0.893,2.443
117,cat__RISK_CAT_1d,0.838,0.838,2.311
83,cat__REGION_RG,-0.795,0.795,0.452
92,cat__REGION_SR,0.790,0.790,2.204
99,cat__REGION_TP,-0.780,0.780,0.458


which registry features are pushing reactivation probability up or down?

Interpret logistic regression coefficients

Goal:
- inspect which features increase or decrease predicted reactivation probability
- use this as a first business interpretation step


How to interpret

For each row:

positive coefficient → pushes prediction toward target = 1

negative coefficient → pushes prediction toward target = 0

larger absolute value → stronger effect

odds_ratio > 1 → increases odds of reactivation

odds_ratio < 1 → decreases odds of reactivation

For example:

coefficient 0.70 → odds ratio exp(0.70) ≈ 2.01

coefficient -0.50 → odds ratio exp(-0.50) ≈ 0.61

One caution:

numeric features were standardized, so their coefficients are per 1 standard deviation

categorical coefficients are for that category’s dummy variable, so treat them as directional signals, not final business truth

Run that block and paste the top 10 positive and top 10 negative coefficients. Then we’ll translate them into plain business language.

In [34]:
# Check support and empirical reactivation rate by category
for col in ["RISK_CAT", "REGION"]:
    print(f"\n=== {col} ===")
    display(
        model_df.groupby(col)["target"]
        .agg(["count", "mean"])
        .sort_values("mean", ascending=False)
    )


=== RISK_CAT ===


,count,mean
RISK_CAT,,
2c,2,1.000
3d,3857,0.907
1d,25,0.880
5d,2039,0.823
4d,156,0.808
T8,678,0.779
4c,29,0.759
2d,48,0.729
2a,1305,0.718



=== REGION ===


,count,mean
REGION,,
SR,17,0.941
GO,22,0.909
NU,70,0.900
LC,43,0.884
KR,7,0.857
...,...,...
2A,0,NaN
50,0,NaN
74,0,NaN
